In [ ]:
from qutip import *
import numpy as np
from scipy.special import jn_zeros

# -----------------------------
# time-dependent drive
# -----------------------------
def h_t(t, args):
    h = args['h']
    h0 = args['h0']
    w = args['omega']
    return h0 + h * np.cos(w * t)

# -----------------------------
# LMG Hamiltonian pieces
# -----------------------------
def h0_ham_lmg(N, n_ph, g, hbar, omega0, Jvalue):

    S = N / 2

    # collective spin operators
    Sx = jmat(S, 'x')
    Sz = jmat(S, 'z')
    Sz2 = Sz * Sz

    dim_spin = int(2*S + 1)
    I_spin = qeye(dim_spin)

    # cavity
    a = destroy(n_ph)
    adag = a.dag()
    I_ph = qeye(n_ph)

    # -------- tensor lift --------
    Sx_full = tensor(Sx, I_ph)
    Sz_full = tensor(Sz, I_ph)
    Sz2_full = tensor(Sz2, I_ph)

    a_full = tensor(I_spin, a)
    adag_full = tensor(I_spin, adag)

    # -------- Hamiltonian parts --------

    # LMG interaction
    H0 = -2/(N-1) * Sz2_full

    # transverse field (time-dependent part will multiply this)
    H1 = 2 * Sx_full

    # cavity energy
    H2 = hbar * omega0 /(N-1)* (adag_full * a_full)

    # light-matter coupling
    H_int = (2 * g / np.sqrt(N)) * (a_full + adag_full) * Sx_full

    return H0, H1, H2, H_int, Sx_full, Sz_full, Sx


def run_dynamics_lmg(args):

    tlist = args['tlist']
    N = args['N']
    n_ph = args['n_ph']
    g = args['g']
    hbar = args['hbar']
    omega0 = args['omega0']
    Jvalue = args['Jvalue']

    H0, H1, H2, H_int, Sx_full, Sz_full, Sx = h0_ham_lmg(N, n_ph, g, hbar, omega0, Jvalue)

    # time-dependent Hamiltonian
    H_td = [H0 + H2 + H_int, [H1, h_t]]

    # initial state (fully polarized + vacuum)

    en, st = Sx.eigenstates()
    a = np.where(np.isclose(en, max(en)))
    psi_spin = st[a[0][0]]
    psi_ph = basis(n_ph, 0)
    psi0 = tensor(psi_spin, psi_ph)
    # evolve
    result = mesolve(H_td,psi0,tlist,[],[Sx_full / N, Sz_full / N],args=args,options=args['opts'])

    
    steady_sx = result.expect[0][-100000:]
    steady_sz = result.expect[1][-100000:]
    op_Sx_avg = np.average(steady_sx)
    op_Sz_avg = np.average(steady_sz)

    return op_Sx_avg, op_Sz_avg

In [ ]:
import matplotlib.pyplot as plt
from multiprocessing import Pool
from tqdm import tqdm

# freezing points
frz = jn_zeros(0, 4)

# parameters
N = 10
hbar = 1.0
h0 = 0
n_ph = 11
#g = 0.5
omega0 = 1.0
Jvalue = 1.0

omega = 90

hs = np.linspace(0, 13, 26) * omega / 4
hs = np.sort(np.append(hs, frz * omega / 4))

# time
ttop = 1000
tlist = np.linspace(0, ttop, ttop * 200 + 1)
opts = Options(nsteps =1e6, atol=1e-10, rtol=1e-8, max_step=0.001)

gvals = [0, 0.1, 0.5]
colors = ['blue', 'red', 'green']
styles = ['-', '--', ':']

for g, c, s  in zip(gvals, colors, styles):
   args_list = [{'omega': omega, 'h0': h0, 'g': g, 'tlist': tlist, 'N': N, 'n_ph': n_ph, 'hbar': hbar, 'omega0': omega0, 'Jvalue': Jvalue, 'h':h, 'opts':opts } 
                for h in hs]

  # parallel run

   p = Pool(processes=10)
   results = p.map(run_dynamics_lmg, tqdm(args_list))

   p.close()
   p.join()

   results = np.array(results)
   Sx_avgs = results[:, 0]
   Sz_avgs = results[:, 1]
   np.savez(f'data/localization_thermalization_N_{N}_nph_{n_ph}_g_{g}.npz', x = 4*hs/omega, sx = Sx_avgs, sz = Sz_avgs)

   # plot
   plt.plot(4*hs/omega, Sx_avgs, color=c, linestyle=s, label=rf'$g={g}$')
   plt.plot(4*hs/omega, Sz_avgs, color=c, linestyle=s)

dt = tlist[1] - tlist[0]
for i in frz:
    plt.axvline(x=i, ls='--', color='green', lw=0.5)

plt.xlabel(r'$4h / \omega$')
plt.ylabel(r'$\langle S_x \rangle$')
plt.legend(frameon=False, loc='upper left')
plt.tight_layout()
plt.savefig(f'plots/Localization_Thermalization_N_{N}_nph_{n_ph}_omega_{omega}_dt_{dt:.4f}.png', dpi=300, bbox_inches='tight')
plt.show()